In [0]:
base_path = "/Volumes/de_workspace26/tejpal_shop/tejpal_raw_volume/"

In [0]:
stream_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/de_workspace26/tejpal_shop/tejpal_schema/orders_stream") \
    .load(base_path)


In [0]:
from pyspark.sql.functions import to_timestamp

stream_df = stream_df.withColumn(
    "order_date", to_timestamp("order_date")
)

In [0]:
stream_df = stream_df.withWatermark("order_date", "10 minutes")

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/de_workspace26/tejpal_shop/tejpal_checkpoints/orders_stream") \
    .trigger(once=True) \
    .table("de_workspace26.tejpal_shop.tejpal_gold_live_orders")

In [0]:
spark.table("de_workspace26.tejpal_shop.tejpal_gold_live_orders").count()

In [0]:
# Cell: Count BEFORE stopping the stream
count_before = spark.table("de_workspace26.tejpal_shop.tejpal_gold_live_orders").count()
print(f"Row count BEFORE restart: {count_before}")

In [0]:
# Cell: Restart the stream (same checkpoint location = no duplicates)
query = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/de_workspace26/tejpal_shop/tejpal_schema/orders_stream") \
    .load(base_path) \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/de_workspace26/tejpal_shop/tejpal_checkpoints/orders_stream") \
    .trigger(once=True) \
    .table("de_workspace26.tejpal_shop.tejpal_gold_live_orders")

import time
time.sleep(15)  # wait for stream to process
print("Stream restarted.")

In [0]:
# Cell: Count AFTER restart — must be same number, proving no duplicates
count_after = spark.table("de_workspace26.tejpal_shop.tejpal_gold_live_orders").count()
print(f"Row count AFTER restart: {count_after}")
print(f"Duplicates added: {count_after - count_before}")
assert count_after == count_before, "ERROR: Duplicates detected!"
print("✅ Checkpoint working correctly - no duplicates written.")
